<a href="https://colab.research.google.com/github/mro9395/risk_zones/blob/main/risk_polygon_population_cluster_sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Citywide high-risk polygons: population, clusters, and a site draw

This notebook screens **all CDMX high-risk polygons** in `susceptibilidad_laderas_all.geojson` against the 2026 WorldPop population-count raster. It retains polygons with estimated population above the editable threshold, clusters nearby retained polygons, and draws five primary clusters plus five reserves across broad geographic quadrants. It does **not** sample cadastre parcels or count households.

WorldPop's raster values are estimated **people per pixel**, not population density. The zonal sum multiplies each pixel count by the fraction of its area covered by a polygon. These are model-based estimates, especially uncertain for small polygons. A positive value is a screening indicator, not proof that a household occupies the polygon. [WorldPop product metadata](https://stac.worldpop.org/collections/MEX/items/mex_pop_2026_CN_100m_R2025A_v1) · [exactextract sum definition](https://isciences.github.io/exactextract/operations.html).

The source GeoJSON is treated as already limited to high-risk polygons. No additional risk-category filter or four-alcaldía clipping is applied. The random draw creates **candidate study sites** for later physical inspection, not an automatically approved fieldwork sample.


In [ ]:
%pip -q install geopandas pyogrio shapely pyproj rasterio exactextract matplotlib

from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import Window, from_bounds, bounds as window_bounds
from shapely.geometry import box
from shapely.ops import unary_union
from exactextract import exact_extract
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

try:
    from google.colab import drive
except ImportError:
    drive = None

# Set input names and the site-draw design here.
RAW = Path('/content/drive/MyDrive/Dissertation/data/raw')
WORK = Path('/content/drive/MyDrive/Dissertation/data/output')
BOUNDARY_FILE_NAME = 'susceptibilidad_laderas_all.geojson'
POPULATION_RASTER_STEM = 'mex_pop_2026_CN_100m_R2025A_v1'
MIN_POLYGON_POPULATION = 0.0  # retain estimates strictly above this value
CLUSTER_GAPS_M = (25, 50, 100, 200)
CLUSTER_DISTANCE_METHOD = 'perimeter'  # 'perimeter' or 'centroid'
DRAW_CLUSTER_GAP_M = 50
N_PRIMARY_CLUSTERS = 5
N_RESERVE_CLUSTERS = 5
RANDOM_SEED = 20261109
METRIC_CRS = 'EPSG:32614'

if drive is not None and not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=False)
if not RAW.exists() or not WORK.exists():
    raise FileNotFoundError(f'Expected existing Drive folders: {RAW} and {WORK}')
if MIN_POLYGON_POPULATION < 0:
    raise ValueError('MIN_POLYGON_POPULATION must be nonnegative.')
if CLUSTER_DISTANCE_METHOD not in {'perimeter', 'centroid'}:
    raise ValueError('CLUSTER_DISTANCE_METHOD must be perimeter or centroid.')
if DRAW_CLUSTER_GAP_M not in CLUSTER_GAPS_M or any(gap <= 0 for gap in CLUSTER_GAPS_M):
    raise ValueError('The draw gap must be one of the positive sensitivity gaps.')
if N_PRIMARY_CLUSTERS < 1 or N_RESERVE_CLUSTERS < 0:
    raise ValueError('Request at least one primary cluster and no negative reserve count.')

def unique_input(filename, suffixes):
    '''Find exactly one named file under RAW; fail on missing or duplicate copies.'''
    matches = sorted(path for path in RAW.rglob('*') if path.is_file()
                     and path.suffix.lower() in suffixes
                     and (path.name.lower() == filename.lower()
                          or path.stem.lower() == filename.lower()))
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected one {filename} under {RAW}; found {matches}')
    return matches[0]

BOUNDARY_PATH = unique_input(BOUNDARY_FILE_NAME, {'.geojson'})
RASTER_PATH = unique_input(POPULATION_RASTER_STEM, {'.tif', '.tiff'})
print('High-risk polygons:', BOUNDARY_PATH)
print('Population raster:', RASTER_PATH)


## 1. Read the citywide boundaries and intersect the raster

Each source feature must be a polygon or multipolygon. Existing `risk_polygon_id` values are preserved when present; otherwise IDs are generated in source-file order and the source row number is retained. Run the notebook again with the same input file to reproduce those IDs. Missing raster coverage is reported separately from a valid zero population estimate.


In [ ]:
source = gpd.read_file(BOUNDARY_PATH)
if source.crs is None:
    raise ValueError(f'{BOUNDARY_PATH.name} has no CRS.')
if source.empty or source.geometry.isna().any() or source.geometry.is_empty.any():
    raise ValueError('The citywide boundary layer is empty or contains missing geometries.')
source = source.reset_index(drop=True).copy()
source['source_feature_index'] = np.arange(1, len(source) + 1)
source['geometry'] = source.geometry.make_valid()
if not source.geometry.geom_type.isin(['Polygon', 'MultiPolygon']).all():
    kinds = source.geometry.geom_type.value_counts().to_dict()
    raise ValueError(f'Expected polygonal high-risk features only; found {kinds}')
if 'risk_polygon_id' in source.columns:
    ids = source.risk_polygon_id.astype(str).str.strip()
    if ids.eq('').any() or source.risk_polygon_id.isna().any() or ids.duplicated().any():
        raise ValueError('risk_polygon_id must be complete and unique in the source file.')
else:
    ids = pd.Series([f'P{i:06d}' for i in source.source_feature_index])

polygons = gpd.GeoDataFrame({
    'risk_polygon_id': ids.to_numpy(),
    'source_feature_index': source.source_feature_index.to_numpy(),
    'geometry': source.geometry.to_numpy(),
}, geometry='geometry', crs=source.crs).to_crs(METRIC_CRS)
polygons['polygon_area_m2'] = polygons.geometry.area
if polygons.polygon_area_m2.le(0).any():
    raise ValueError('Risk polygons include zero-area geometries after repair.')

with rasterio.open(RASTER_PATH) as raster:
    if raster.count != 1 or raster.crs is None or raster.nodata is None:
        raise ValueError('Expected a single-band georeferenced population raster with nodata.')
    raster_crs = raster.crs
    # exactextract performs an area-fraction-weighted sum of people-per-cell values.
    polygon_stats = exact_extract(
        raster, polygons.to_crs(raster_crs),
        ['population_estimate=sum', 'valid_cell_coverage=count'],
        include_cols='risk_polygon_id', output='pandas',
    )

if polygon_stats.risk_polygon_id.duplicated().any() or len(polygon_stats) != len(polygons):
    raise AssertionError('Raster extraction did not return one row per source polygon.')
polygons = polygons.merge(polygon_stats, on='risk_polygon_id', validate='one_to_one')
polygons['valid_cell_coverage'] = polygons.valid_cell_coverage.fillna(0)
polygons['population_estimate'] = pd.to_numeric(polygons.population_estimate, errors='coerce')
if polygons.population_estimate.dropna().lt(-1e-6).any():
    raise ValueError('The population raster has negative values beyond nodata.')
polygons['screening_status'] = np.select(
    [polygons.valid_cell_coverage.le(0) | polygons.population_estimate.isna(),
     polygons.population_estimate.le(MIN_POLYGON_POPULATION)],
    ['no_raster_data', 'below_or_equal_threshold'], default='eligible',
)
eligible_polygons = polygons[polygons.screening_status.eq('eligible')].copy().reset_index(drop=True)
if eligible_polygons.empty:
    raise ValueError('No polygons passed the population screen. Inspect raster coverage and threshold.')
print('Source polygons:', len(polygons))
print('Population-screening status:', polygons.screening_status.value_counts().to_dict())
display(polygons.drop(columns='geometry').head(10))

polygons.to_crs(4326).to_file(WORK / 'citywide_risk_polygon_population_screen.geojson', driver='GeoJSON')
polygons.drop(columns='geometry').to_csv(WORK / 'citywide_risk_polygon_population_screen.csv', index=False)
eligible_polygons.to_crs(4326).to_file(WORK / 'citywide_population_eligible_polygons.geojson', driver='GeoJSON')


## 2. Cluster retained polygons

Polygons connect when their edges are at most the specified gap apart. Connected chains form one cluster; member polygons are dissolved before estimating **cluster population** so overlapping risk polygons cannot double count raster cells. The four gap scenarios are screening alternatives. Only `DRAW_CLUSTER_GAP_M` feeds the random draw; changing it changes the draw frame. The optional centroid metric is retained for sensitivity.


In [ ]:
def make_clusters(frame, gap_m, method):
    '''Build connected components from perimeter or centroid distance.'''
    work = frame.sort_values('risk_polygon_id').reset_index(drop=True).copy()
    comparison = work.geometry.centroid if method == 'centroid' else work.geometry
    expanded = gpd.GeoSeries(comparison.buffer(gap_m / 2), crs=work.crs)
    parent = list(range(len(work)))

    def find(item):
        '''Find a component root while compressing the union-find path.'''
        while parent[item] != item:
            parent[item] = parent[parent[item]]
            item = parent[item]
        return item

    pairs = expanded.sindex.query(expanded, predicate='intersects')
    for left, right in zip(*pairs):
        left, right = int(left), int(right)
        if left < right:
            root_left, root_right = find(left), find(right)
            if root_left != root_right:
                parent[root_right] = root_left

    work['component_root'] = [find(i) for i in range(len(work))]
    records, members = [], []
    # Sorting by the smallest member ID stabilizes labels for the same source file.
    groups = sorted((group for _, group in work.groupby('component_root')),
                    key=lambda group: group.risk_polygon_id.min())
    for number, group in enumerate(groups, start=1):
        cluster_id = f'{method[:3].upper()}_{gap_m:03d}m_C{number:05d}'
        ids = sorted(group.risk_polygon_id.tolist())
        records.append({
            'cluster_id': cluster_id,
            'cluster_gap_m': gap_m,
            'cluster_distance_method': method,
            'risk_polygon_count': len(ids),
            'member_polygon_ids': ','.join(ids),
            'geometry': unary_union(group.geometry.tolist()),
        })
        members.extend({'cluster_id': cluster_id, 'risk_polygon_id': polygon_id,
                        'cluster_gap_m': gap_m} for polygon_id in ids)
    return gpd.GeoDataFrame(records, geometry='geometry', crs=frame.crs), pd.DataFrame(members)

cluster_parts, membership_parts = [], []
for gap in CLUSTER_GAPS_M:
    one_scenario, one_membership = make_clusters(eligible_polygons, gap, CLUSTER_DISTANCE_METHOD)
    with rasterio.open(RASTER_PATH) as raster:
        cluster_stats = exact_extract(
            raster, one_scenario.to_crs(raster.crs),
            ['cluster_population_estimate=sum', 'valid_cell_coverage=count'],
            include_cols='cluster_id', output='pandas',
        )
    one_scenario = one_scenario.merge(cluster_stats, on='cluster_id', validate='one_to_one')
    if one_scenario.cluster_population_estimate.isna().any():
        raise ValueError(f'Cluster population extraction has missing values at {gap} m.')
    one_scenario['cluster_area_m2'] = one_scenario.geometry.area
    cluster_parts.append(one_scenario)
    membership_parts.append(one_membership)
    print(f'{gap} m {CLUSTER_DISTANCE_METHOD}: {len(one_scenario):,} clusters')

clusters = gpd.GeoDataFrame(pd.concat(cluster_parts, ignore_index=True),
                            geometry='geometry', crs=METRIC_CRS)
membership = pd.concat(membership_parts, ignore_index=True)
if membership.duplicated(['cluster_gap_m', 'risk_polygon_id']).any():
    raise AssertionError('A polygon belongs to more than one cluster in one gap scenario.')
if not set(membership.risk_polygon_id).eq(set(eligible_polygons.risk_polygon_id)):
    raise AssertionError('A population-eligible polygon is missing from clustering.')
clusters.to_crs(4326).to_file(WORK / 'citywide_population_risk_clusters.geojson', driver='GeoJSON')
clusters.drop(columns='geometry').to_csv(WORK / 'citywide_population_risk_clusters.csv', index=False)
membership.to_csv(WORK / 'citywide_population_cluster_membership.csv', index=False)
display(clusters.drop(columns='geometry').groupby('cluster_gap_m').agg(
    clusters=('cluster_id', 'size'), polygons=('risk_polygon_count', 'sum'),
    median_population=('cluster_population_estimate', 'median')))


## 3. Draw primary and reserve sites across four broad geographic groups

The geographic groups are west/east and south/north quadrants split at the median **eligible-cluster representative-point** coordinates for the chosen gap. This creates broad, data-driven spatial strata rather than quotas for 16 alcaldías. Within each group, clusters are shuffled with the fixed seed, then drawn round-robin across groups. The first five are primary candidates; the next five are reserves. If fewer than ten clusters exist, stop and revise the requested counts explicitly. Random selection does not establish physical comparability; review terrain, ravines, roads, and site access afterward.


In [ ]:
draw_frame = clusters[clusters.cluster_gap_m.eq(DRAW_CLUSTER_GAP_M)].copy().reset_index(drop=True)
requested = N_PRIMARY_CLUSTERS + N_RESERVE_CLUSTERS
if len(draw_frame) < requested:
    raise ValueError(f'Only {len(draw_frame)} clusters exist at {DRAW_CLUSTER_GAP_M} m; '
                     f'{requested} primary + reserve sites were requested.')
anchors = draw_frame.geometry.representative_point()
split_x, split_y = float(anchors.x.median()), float(anchors.y.median())
draw_frame['geographic_group'] = (
    np.where(anchors.y >= split_y, 'north', 'south') + '_' +
    np.where(anchors.x >= split_x, 'east', 'west')
)
print('Draw-frame clusters by geographic group:')
display(draw_frame.geographic_group.value_counts().sort_index().rename('clusters').to_frame())

def spatially_stratified_order(frame, seed):
    '''Shuffle clusters within each quadrant, then alternate among quadrants.'''
    rng = np.random.default_rng(seed)
    groups = {}
    for name, group in frame.groupby('geographic_group', sort=True):
        ids = sorted(group.cluster_id.tolist())
        groups[name] = list(rng.permutation(ids))
    group_order = list(rng.permutation(sorted(groups)))
    result = []
    while any(groups.values()):
        for name in group_order:
            if groups[name]:
                result.append(groups[name].pop())
    if len(result) != len(frame) or len(set(result)) != len(result):
        raise AssertionError('Spatial draw omitted or duplicated a cluster.')
    return result

ordered_ids = spatially_stratified_order(draw_frame, RANDOM_SEED)
rank_lookup = {cluster_id: rank for rank, cluster_id in enumerate(ordered_ids, 1)}
draw_frame['random_order'] = draw_frame.cluster_id.map(rank_lookup)
draw_frame['selection_status'] = np.select(
    [draw_frame.random_order.le(N_PRIMARY_CLUSTERS),
     draw_frame.random_order.le(requested)],
    ['primary', 'reserve'], default='not_selected',
)
selected_clusters = draw_frame[~draw_frame.selection_status.eq('not_selected')].copy()
selected_clusters = selected_clusters.sort_values('random_order')
selected_links = membership[
    membership.cluster_gap_m.eq(DRAW_CLUSTER_GAP_M) &
    membership.cluster_id.isin(selected_clusters.cluster_id)
].merge(selected_clusters[['cluster_id', 'geographic_group', 'selection_status',
                           'random_order']], on='cluster_id', validate='many_to_one')
selected_polygons = eligible_polygons.merge(
    selected_links.drop(columns='cluster_gap_m'), on='risk_polygon_id', validate='one_to_one'
)
if len(selected_polygons) != selected_clusters.risk_polygon_count.sum():
    raise AssertionError('Selected polygon membership does not match the selected clusters.')

draw_frame.to_crs(4326).to_file(WORK / 'citywide_population_cluster_draw_frame.geojson', driver='GeoJSON')
draw_frame.drop(columns='geometry').to_csv(WORK / 'citywide_population_cluster_draw_frame.csv', index=False)
selected_clusters.to_crs(4326).to_file(WORK / 'citywide_population_selected_clusters.geojson', driver='GeoJSON')
selected_clusters.drop(columns='geometry').to_csv(WORK / 'citywide_population_selected_clusters.csv', index=False)
selected_polygons.to_crs(4326).to_file(WORK / 'citywide_population_selected_boundary_polygons.geojson', driver='GeoJSON')
manifest = {
    'boundary_file': str(BOUNDARY_PATH),
    'population_raster': str(RASTER_PATH),
    'population_units': 'estimated people per raster cell',
    'population_method': 'sum of cell counts weighted by cell fraction inside each polygon',
    'minimum_polygon_population_exclusive': MIN_POLYGON_POPULATION,
    'cluster_gaps_m': list(CLUSTER_GAPS_M),
    'cluster_distance_method': CLUSTER_DISTANCE_METHOD,
    'draw_cluster_gap_m': DRAW_CLUSTER_GAP_M,
    'primary_clusters': N_PRIMARY_CLUSTERS,
    'reserve_clusters': N_RESERVE_CLUSTERS,
    'random_seed': RANDOM_SEED,
    'geographic_groups': 'four quadrants split at median eligible-cluster representative-point x/y',
    'quadrant_split_x_epsg32614': split_x,
    'quadrant_split_y_epsg32614': split_y,
    'source_polygon_count': len(polygons),
    'eligible_polygon_count': len(eligible_polygons),
    'draw_frame_cluster_count': len(draw_frame),
}
(WORK / 'citywide_population_cluster_manifest.json').write_text(
    json.dumps(manifest, indent=2), encoding='utf-8'
)
display(selected_clusters[['random_order', 'selection_status', 'cluster_id',
                           'geographic_group', 'risk_polygon_count',
                           'cluster_population_estimate']].reset_index(drop=True))


## 4. Preview source data, screening, and selected sites

The overview uses a downsampled raster **for display only**; extraction above uses the original raster. Red outlines are primary clusters; orange outlines are reserves. The close-up maps show the raster, all local risk polygons, and the selected cluster boundary. The plot and CSVs make the sampling frame and random order auditable before downloading GeoJSONs.


In [ ]:
def raster_preview(ax, metric_bounds, max_side=950):
    '''Read only a bounded, downsampled raster window for a map preview.'''
    metric_box = gpd.GeoSeries([box(*metric_bounds)], crs=METRIC_CRS)
    left, bottom, right, top = metric_box.to_crs(raster_crs).total_bounds
    with rasterio.open(RASTER_PATH) as raster:
        window = from_bounds(left, bottom, right, top, raster.transform)
        window = window.intersection(Window(0, 0, raster.width, raster.height))
        width, height = max(1, round(window.width)), max(1, round(window.height))
        scale = min(1, max_side / max(width, height))
        out_shape = (max(1, round(height * scale)), max(1, round(width * scale)))
        image = raster.read(1, window=window, out_shape=out_shape,
                            resampling=Resampling.average, masked=True)
        extent = window_bounds(window, raster.transform)
    # Log colour scaling keeps both sparse and dense cells visible.
    image = np.ma.log1p(np.ma.maximum(image, 0))
    return ax.imshow(image, extent=extent, origin='upper', cmap='viridis', alpha=0.75)

overview_bounds = polygons.total_bounds
pad = 1000
overview_bounds = (overview_bounds[0]-pad, overview_bounds[1]-pad,
                   overview_bounds[2]+pad, overview_bounds[3]+pad)
fig, ax = plt.subplots(figsize=(11, 10))
image = raster_preview(ax, overview_bounds)
polygons.to_crs(raster_crs).boundary.plot(ax=ax, color='#8a8a8a', linewidth=0.15, alpha=0.6)
eligible_polygons.to_crs(raster_crs).boundary.plot(ax=ax, color='#d9f0a3', linewidth=0.25)
for status, colour in [('reserve', '#fe9929'), ('primary', '#de2d26')]:
    subset = selected_clusters[selected_clusters.selection_status.eq(status)]
    subset.to_crs(raster_crs).boundary.plot(ax=ax, color=colour, linewidth=2)
ax.set_title('CDMX source polygons, population raster, and selected clusters')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
fig.colorbar(image, ax=ax, shrink=0.65,
             label='log(1 + average estimated people per source cell)')
ax.legend(handles=[
    Patch(color='#8a8a8a', label='All high-risk polygon boundaries'),
    Patch(color='#d9f0a3', label='Population-eligible boundaries'),
    Line2D([0], [0], color='#de2d26', lw=2, label='Primary clusters'),
    Line2D([0], [0], color='#fe9929', lw=2, label='Reserve clusters'),
], loc='best')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
status_counts = polygons.screening_status.value_counts().reindex(
    ['eligible', 'below_or_equal_threshold', 'no_raster_data'], fill_value=0)
axes[0].barh(status_counts.index, status_counts.values,
             color=['#31a354', '#bdbdbd', '#de2d26'])
axes[0].invert_yaxis(); axes[0].set_xlabel('High-risk polygons')
axes[0].set_title('Population screening')
for row in selected_clusters.itertuples(index=False):
    colour = '#de2d26' if row.selection_status == 'primary' else '#fe9929'
    axes[1].barh(f'{row.random_order}. {row.cluster_id}', row.cluster_population_estimate,
                 color=colour)
axes[1].invert_yaxis(); axes[1].set_xlabel('Estimated population inside dissolved cluster')
axes[1].set_title('Drawn primary and reserve clusters')
plt.tight_layout(); plt.show()

map_rows = int(np.ceil(len(selected_clusters) / 2))
fig, axes = plt.subplots(map_rows, 2, figsize=(15, 5 * map_rows), squeeze=False)
for ax, row in zip(axes.flat, selected_clusters.itertuples(index=False)):
    minx, miny, maxx, maxy = row.geometry.bounds
    pad = max(250, 0.12 * max(maxx-minx, maxy-miny))
    bounds = (minx-pad, miny-pad, maxx+pad, maxy+pad)
    raster_preview(ax, bounds, max_side=600)
    local_ids = list(polygons.sindex.query(box(*bounds), predicate='intersects'))
    polygons.iloc[local_ids].to_crs(raster_crs).boundary.plot(
        ax=ax, color='#cccccc', linewidth=0.5)
    members = selected_polygons[selected_polygons.cluster_id.eq(row.cluster_id)]
    members.to_crs(raster_crs).boundary.plot(ax=ax, color='#ffffff', linewidth=1.1)
    gpd.GeoSeries([row.geometry], crs=METRIC_CRS).to_crs(raster_crs).boundary.plot(
        ax=ax, color='#de2d26' if row.selection_status == 'primary' else '#fe9929',
        linewidth=2)
    ax.set_title(f'{row.random_order}. {row.selection_status} | {row.cluster_id} | '
                 f'{row.risk_polygon_count} polygons | ~{row.cluster_population_estimate:,.0f} people')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
for ax in list(axes.flat)[len(selected_clusters):]:
    ax.axis('off')
plt.tight_layout(); plt.show()


## Outputs and interpretation

- `citywide_risk_polygon_population_screen.csv/.geojson`: every source polygon, estimated population, valid raster-cell coverage, and reason for inclusion/exclusion.
- `citywide_population_eligible_polygons.geojson`: polygons strictly above the population threshold.
- `citywide_population_risk_clusters.csv/.geojson`: dissolved clusters for all four gap scenarios and their population estimates.
- `citywide_population_cluster_membership.csv`: polygon-to-cluster link for each scenario.
- `citywide_population_cluster_draw_frame.csv/.geojson`: all clusters at the chosen draw gap, with geographic group, random order, and selection status.
- `citywide_population_selected_clusters.csv/.geojson`: five primary plus five reserve dissolved cluster polygons.
- `citywide_population_selected_boundary_polygons.geojson`: individual source high-risk polygons belonging to those ten clusters.
- `citywide_population_cluster_manifest.json`: exact source filenames, settings, counts, seed, and geographic split.

The source is the specified high-risk GeoJSON and the 2026 WorldPop raster only. The notebook does not use the four-alcaldía cadastre or vulnerable-lot data. Cluster population is recomputed on dissolved geometry to avoid double counting overlapping polygons. A polygon can belong to one cluster **per gap scenario**, so comparisons across scenarios are sensitivity analyses, not additive counts. Inspect each selected site's physical boundary and eligible parcels before treating it as a dissertation sampling location.
